In [9]:
import sqlite3
import pandas as pd
import logging
import os
from ingestion_db import ingest_db

os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    filename="logs/get_vendor_summary.log",
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode='a'
)

def create_vendor_summary(conn):
    '''Merge tables to get overall vendor summary'''

    vendor_sales_summary = pd.read_sql_query("""
    -- (Your SQL remains unchanged)
    WITH FreightSummary AS (
        SELECT VendorNumber,
               SUM(Freight) AS FreightCost
        FROM vendor_invoice
        GROUP BY VendorNumber
    ),
    PurchaseSummary AS (
        SELECT
            p.VendorNumber,
            p.VendorName,
            p.Brand,
            p.Description,
            p.PurchasePrice,
            pp.Price AS ActualPrice,
            pp.Volume,
            SUM(p.Quantity) AS TotalPurchaseQuantity,
            SUM(p.Dollars) AS TotalPurchaseDollars
        FROM purchases p
        JOIN purchase_prices pp
            ON p.Brand = pp.Brand
            AND p.VendorNumber = pp.VendorNumber
        WHERE p.PurchasePrice > 0
        GROUP BY 
            p.VendorNumber, 
            p.VendorName, 
            p.Brand, 
            p.Description, 
            p.PurchasePrice, 
            pp.Price, 
            pp.Volume
    ),
    SalesSummary AS (
        SELECT
            VendorNo,
            Brand,
            SUM(SalesQuantity) AS TotalSalesQuantity,
            SUM(SalesDollars) AS TotalSalesDollars,
            SUM(ExciseTax) AS TotalExciseTax
        FROM sales
        GROUP BY VendorNo, Brand
    )
    SELECT
        ps.VendorNumber,
        ps.VendorName,
        ps.Brand,
        ps.Description,
        ps.PurchasePrice,
        ps.ActualPrice,
        ps.Volume,
        ps.TotalPurchaseQuantity,
        ps.TotalPurchaseDollars,
        ss.TotalSalesQuantity,
        ss.TotalSalesDollars,
        ss.TotalExciseTax,
        fs.FreightCost
    FROM PurchaseSummary ps
    LEFT JOIN SalesSummary ss
        ON ps.VendorNumber = ss.VendorNo
        AND ps.Brand = ss.Brand
    LEFT JOIN FreightSummary fs
        ON ps.VendorNumber = fs.VendorNumber
    ORDER BY ps.TotalPurchaseDollars DESC
    """, conn)

    logging.info("Vendor summary created successfully")
    return vendor_sales_summary


def clean_data(df):
    '''Clean and enhance vendor summary data'''

    df['Volume'] = df['Volume'].astype(float)
    df.fillna(0, inplace=True)

    df['VendorName'] = df['VendorName'].str.strip()
    df['Description'] = df['Description'].str.strip()

    df['GrossProfit'] = df['TotalSalesDollars'] - df['TotalPurchaseDollars']
    df['ProfitMargin'] = (
        df['GrossProfit'] / df['TotalSalesDollars']
    ).replace([float('inf'), -float('inf')], 0) * 100

    df['StockTurnover'] = df['TotalSalesQuantity'] / df['TotalPurchaseQuantity']
    df['SalesToPurchaseRatio'] = df['TotalSalesDollars'] / df['TotalPurchaseDollars']

    logging.info("Data cleaned and KPI columns created")

    return df


if __name__ == "__main__":

    conn = sqlite3.connect('inventory.db')

    logging.info("Creating Vendor Summary Table...")
    summary_df = create_vendor_summary(conn)
    logging.info("Vendor Summary Preview:\n%s", summary_df.head())

    logging.info("Cleaning Data...")
    clean_df = clean_data(summary_df)
    logging.info("Clean Data Preview:\n%s", clean_df.head())

    logging.info("Ingesting data...")
    ingest_db(clean_df, 'vendor_sales_summary', conn)

    logging.info("Process Completed Successfully")
    conn.close()

SyntaxError: unterminated string literal (detected at line 120) (1902651378.py, line 120)